# 第7章　债券组合管理

[![在 Colab 打开](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/albertandking/fixed-income/blob/main/notebooks/ch07_portfolio.ipynb) [![在 Binder 打开](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/albertandking/fixed-income/main?labpath=notebooks/ch07_portfolio.ipynb)

复现例7.1（单期免疫）、例7.2（哑铃久期匹配）、例7.3（现金流匹配 LP），并汇总组合层风险。


In [ ]:
# 自举单元：在 Colab/Binder 上自动安装本书复用包 fi；本地运行时自动跳过。
import importlib.util, sys, subprocess
if importlib.util.find_spec('fi') is None:
    if 'google.colab' in sys.modules:
        subprocess.run(['git', 'clone', '--depth', '1',
                        'https://github.com/albertandking/fixed-income.git', '/content/fi-book'], check=False)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '/content/fi-book'], check=False)
    else:
        print('提示：请在仓库根目录执行 `uv sync --extra all` 后再运行本 notebook。')


In [ ]:
import numpy as np
from fi import portfolio as pf, risk, plotting
from fi.cashflow import make_cashflows
from fi.pricing import price_bond
plotting.use_chinese_style()


## 例7.1　单期免疫：久期 = 持有期


In [ ]:
cfs, ts = make_cashflows(0.03, 6, freq=1, face=100)
y0 = 0.03
P0 = price_bond(cfs, ts, y0, 1)
H = risk.macaulay_duration(cfs, ts, y0, 1)
target = P0 * (1 + y0) ** H
print(f'6yr 3% 平价债: 价格={P0:.4f}, 麦考利久期={H:.4f}年')
print(f'设持有期 H = 久期, 目标终值 = {target:.4f}')
for dy in (-0.01, -0.005, 0.0, 0.005, 0.01):
    v = sum(cf * (1 + y0 + dy) ** (H - t) for cf, t in zip(cfs, ts))
    print(f'  Δy={dy*100:+.1f}%: 实现终值={v:.4f}  ({"=目标" if dy==0 else ">目标"})')


### 免疫的 U 形：实现终值关于 Δy（编程实验 7）


In [ ]:
dys = np.linspace(-0.02, 0.02, 81)
vals = [sum(cf*(1+y0+dy)**(H-t) for cf,t in zip(cfs,ts)) for dy in dys]
fig, ax = plotting.new_axes()
ax.plot(dys*100, vals)
ax.axhline(target, ls=':', color='gray', label=f'目标终值 {target:.2f}')
ax.axvline(0, ls=':', color='gray')
ax.set_xlabel('利率平行移动 Δy (%)'); ax.set_ylabel(f'H={H:.2f}年 时点实现终值')
ax.set_title('图7-1　单期免疫：久期=持有期时财富被锁定'); ax.legend()
fig.tight_layout()


## 例7.2　哑铃久期匹配


In [ ]:
ws, wl = pf.two_asset_immunization(d_short=1.95, d_long=8.78, d_target=7.0)
print(f'2yr(D=1.95) 权重 = {ws:.4f}')
print(f'10yr(D=8.78) 权重 = {wl:.4f}')
print(f'校验组合久期 = {ws*1.95 + wl*8.78:.4f}  (目标 7)')
# 组合久期（市值加权）
print('等市值三券组合久期 =', round(pf.portfolio_duration([1000,1000,1000], [1.95,4.70,8.78]), 4))


## 例7.3　现金流匹配（线性规划）


In [ ]:
liab = [100, 100, 100]
bond_cf = np.array([[102, 0, 0],     # 1yr
                    [5, 105, 0],     # 2yr 5%
                    [4, 4, 104]], dtype=float)   # 3yr 4%
prices = [100.0, 101.0, 100.5]
res = pf.cash_flow_match(liab, bond_cf, prices)
print('求解成功 :', res['success'])
print('最小成本 :', round(res['cost'], 4))
print('各债份数 :', np.round(res['units'], 4))
cover = bond_cf.T @ res['units']
print('各期资产现金流 :', np.round(cover, 4), ' >= 负债', liab)


### 哑铃 vs 子弹的凸性（编程实验 9 铺垫）

久期都=7，但哑铃凸性更高 —— 免疫更稳健的根源。


In [ ]:
# 子弹：约 7 年久期的单只债（用 8yr 3% 近似）
cf_b, t_b = make_cashflows(0.03, 8, freq=1, face=100)
conv_bullet = risk.convexity(cf_b, t_b, 0.03, 1)
# 哑铃：2yr 与 10yr 按权重组合的凸性（市值加权近似）
cf2, t2 = make_cashflows(0.0195, 2, freq=2, face=100)
cf10, t10 = make_cashflows(0.0255, 10, freq=2, face=100)
conv2 = risk.convexity(cf2, t2, 0.0195, 2)
conv10 = risk.convexity(cf10, t10, 0.0255, 2)
conv_barbell = ws*conv2 + wl*conv10
print(f'子弹(8yr) 凸性  ≈ {conv_bullet:.2f}')
print(f'哑铃(2/10yr) 凸性 ≈ {conv_barbell:.2f}  -> 哑铃凸性更高，免疫更稳健')


---

> 小结：`fi.portfolio` 提供组合久期、久期匹配（免疫）与现金流匹配（LP）。
> 免疫一阶对冲、需再平衡、哑铃凸性占优；现金流匹配完全消除利率风险但成本更高。
